# Soil processing — Andrea

Independent working version. **Do not edit or overwrite the original soils notebook,
its CSV/GPKG outputs, or the main integration.** This notebook reads the raw SSURGO
survey polygons and map-unit aggregate tables directly; original outputs are used only
for comparison. It saves only `soil_features_andrea.csv`.

The output has one row per current well ID (1,993 expected), with **500 m and 1000 m**
features in separate columns. Soil attributes are static in this dataset: join to nitrate
measurements on `gm_well_id`, not crop year. All 5,280 nitrate measurements must remain.
Different IDs at the same coordinates remain separate wells.

Set `ERDOS_DATA_ROOT` to the shared `Data/Erdos Project` folder if it is not in the default location. The raw inputs are `raw/soils` (five SSURGO survey folders) and `raw/Crops` (the Salinas HUC8 boundary geodatabase). The repository also needs the cleaned nitrate CSV, well-buffer GeoPackage, and original soil comparison notebook/CSV/GeoPackage.

The notebook defaults to `notebooks/soils/soil_features_andrea.csv` and stops before processing if that output already exists. Set `ANDREA_SOIL_OUTPUT` to a new path to save a separate run.

In [1]:
from pathlib import Path
import hashlib
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import shapely
from pyproj import Transformer
from IPython.display import display

REPO = next(
    (path.resolve() for path in [Path.cwd(), *Path.cwd().parents]
     if (path / 'notebooks/groundwater/groundwater_nitrate_clean.csv').is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Run this notebook inside the cloned project repository.")

DATA_ROOT = Path(os.environ.get(
    'ERDOS_DATA_ROOT', str(REPO.parent.parent / 'Data' / 'Erdos Project')
)).expanduser().resolve()
RAW_SOILS = DATA_ROOT / 'raw' / 'soils'
RAW_CROPS = DATA_ROOT / 'raw' / 'Crops'
NITRATE_PATH = REPO / 'notebooks/groundwater/groundwater_nitrate_clean.csv'
BUFFER_PATH = REPO / 'notebooks/groundwater/groundwater_well_buffers_500m.gpkg'
ORIGINAL_NOTEBOOK = REPO / 'notebooks/soils/Soil_data_cleanup_team.ipynb'
ORIGINAL_CSV = REPO / 'notebooks/soils/soils_clean.csv'
ORIGINAL_GPKG = REPO / 'notebooks/soils/soils_huc8_attributes.gpkg'
DEFAULT_OUTPUT = REPO / 'notebooks/soils/soil_features_andrea.csv'
OUTPUT_PATH = Path(os.environ.get('ANDREA_SOIL_OUTPUT', str(DEFAULT_OUTPUT))).expanduser()
CRS = 3310
RADII = (500, 1000)
NUMERIC = ['slopegradwta', 'brockdepmin', 'aws0150wta']
CATEGORICAL = ['drclassdcd', 'hydgrpdcd']
ATTRIBUTES = NUMERIC + CATEGORICAL

def require(condition, message):
    if not condition:
        raise ValueError(message)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

protected_paths = [NITRATE_PATH, BUFFER_PATH, ORIGINAL_NOTEBOOK, ORIGINAL_CSV,
                   ORIGINAL_GPKG, REPO / 'data/soils/soils_huc8_attributes.gpkg']
protected_hashes = {str(p): file_hash(p) for p in protected_paths if p.exists()}
require(OUTPUT_PATH.resolve() not in [p.resolve() for p in protected_paths],
        'The output path points to a protected original input.')
required_inputs = [NITRATE_PATH, BUFFER_PATH, ORIGINAL_NOTEBOOK, ORIGINAL_CSV, ORIGINAL_GPKG]
missing_inputs = [path for path in required_inputs if not path.is_file()]
require(not missing_inputs, 'Missing repository inputs: ' + ', '.join(map(str, missing_inputs)))
require(RAW_SOILS.is_dir(), f'Missing raw SSURGO folder: {RAW_SOILS}')
require(RAW_CROPS.is_dir(), f'Missing raw DWR/NHD folder: {RAW_CROPS}')
require(not OUTPUT_PATH.exists(),
        f'Output already exists; choose a new ANDREA_SOIL_OUTPUT path: {OUTPUT_PATH}')

## 1. Current wells and full metric buffers

Use the current cleaned nitrate table, not the older 3,245-well buffer inventory as the
well list. Build both radii from the same coordinates in EPSG:3310 (meters).
The 500 m geometry is checked against the existing buffer file for matching IDs.
Buffers remain whole across watershed and survey boundaries.

In [2]:
observations = pd.read_csv(NITRATE_PATH, dtype={'gm_well_id': 'string'},
                           parse_dates=['gm_samp_collection_date'])
require(len(observations) == 5280 and observations.gm_well_id.nunique() == 1993,
        'The nitrate input changed; review the well counts.')
require(not observations.duplicated().any(), 'Duplicate nitrate rows.')
coords = observations[['gm_well_id', 'gm_latitude', 'gm_longitude']].drop_duplicates()
require(coords.notna().all().all() and coords.gm_well_id.is_unique,
        'Missing or ambiguous coordinates for a well ID.')
wells = gpd.GeoDataFrame(coords, geometry=gpd.points_from_xy(
    coords.gm_longitude, coords.gm_latitude), crs=4326).to_crs(CRS)
buffers = {}
for radius in RADII:
    layer = wells[['gm_well_id', 'geometry']].copy()
    layer.geometry = layer.buffer(radius, quad_segs=16)
    require(layer.is_valid.all(), 'Invalid well buffers.')
    require(np.allclose(layer.area, np.pi * radius**2 * 0.9983943930356184),
            'Unexpected buffer area or units.')
    buffers[radius] = layer
stored = gpd.read_file(BUFFER_PATH).set_index('gm_well_id')
fresh = buffers[500].set_index('gm_well_id')
require(stored.index.is_unique and set(fresh.index) <= set(stored.index),
        'Existing buffer IDs do not cover the current wells.')
old = stored.loc[fresh.index]
require(old.crs.to_epsg() == CRS, 'Unexpected existing buffer CRS.')
buffer_difference_m2 = fresh.geometry.symmetric_difference(old.geometry, align=False).area
require(buffer_difference_m2.max() < 0.01, '500 m buffers differ from the existing geometry.')
print(f'{len(observations):,} measurements; {len(wells):,} well IDs; '
      f'{coords.drop_duplicates(["gm_latitude", "gm_longitude"]).shape[0]:,} coordinate pairs.')
print('Maximum 500 m buffer difference (m2):', buffer_difference_m2.max())

5,280 measurements; 1,993 well IDs; 1,984 coordinate pairs.
Maximum 500 m buffer difference (m2): 0.0


## 2. Read raw map-unit attributes using the SSURGO metadata

Column names and units come from each survey's own `mstabcol.txt`.
No component table is required: `muaggatt` already contains the same map-unit attributes
used by the original notebook. Conflicting duplicate MUKEY values stop processing.

- `slopegradwta`: map-unit weighted slope, **percent**.
- `brockdepmin`: map-unit minimum bedrock depth, **cm**. Its buffer mean is a mean of
  map-unit minimum depths, not a measured depth at the well. Missing values remain unknown.
- `aws0150wta`: available water storage in the top 150 cm, **cm of water**.
- `drclassdcd`, `hydgrpdcd`: categorical drainage class and hydrologic group.

In [3]:
attribute_tables = []
metadata_tables = []
for survey in sorted(RAW_SOILS.glob('CA*')):
    metadata_path = survey / 'tabular/mstabcol.txt'
    table_path = survey / 'tabular/muaggatt.txt'
    require(metadata_path.exists() and table_path.exists(), f'Missing tabular files: {survey.name}')
    metadata = pd.read_csv(metadata_path, sep='|', header=None, dtype='string')
    schema = metadata[metadata[0].eq('muaggatt')].copy()
    schema['position'] = pd.to_numeric(schema[1])
    names = schema.sort_values('position')[2].tolist()
    table = pd.read_csv(table_path, sep='|', header=None, names=names, dtype='string')
    require({'mukey', *ATTRIBUTES} <= set(table), 'Missing required map-unit attributes.')
    require(table.mukey.notna().all(), 'Missing MUKEY in muaggatt.')
    attribute_tables.append(table[['mukey', *ATTRIBUTES]])
    definitions = schema[schema[2].isin(ATTRIBUTES)][[2, 4, 11, 13]].copy()
    definitions.columns = ['attribute', 'label', 'unit', 'definition']
    definitions.insert(0, 'survey', survey.name)
    metadata_tables.append(definitions)

attributes = pd.concat(attribute_tables, ignore_index=True).drop_duplicates()
require(not attributes.mukey.duplicated().any(), 'Conflicting MUKEY rows across raw surveys.')
attributes = attributes.rename(columns={'mukey': 'MUKEY'})
for name in NUMERIC:
    numeric = pd.to_numeric(attributes[name], errors='coerce')
    require(not (attributes[name].notna() & numeric.isna()).any(), f'Unparsed numeric values: {name}')
    require(numeric.dropna().ge(0).all(), f'Negative soil attribute: {name}')
    attributes[name] = numeric.astype(float)
for name in CATEGORICAL:
    attributes[name] = attributes[name].str.strip().replace('', pd.NA)
definitions = pd.concat(metadata_tables, ignore_index=True)
display(definitions.drop_duplicates(['attribute', 'unit'])[['attribute', 'label', 'unit']])
print('Raw unique map units:', len(attributes))

# Compare attributes only; original outputs are never used to build Andrea features.
original_attributes = pd.read_csv(ORIGINAL_CSV, dtype={'MUKEY': 'string'})
original_attributes = original_attributes[['MUKEY', *ATTRIBUTES]].drop_duplicates()
require(original_attributes.MUKEY.is_unique, 'Conflicting MUKEY values in the original CSV.')
attribute_comparison = original_attributes.merge(attributes, on='MUKEY', how='left',
                                                validate='one_to_one', suffixes=('_original', '_raw'),
                                                indicator=True)
require(attribute_comparison._merge.eq('both').all(), 'An original MUKEY is missing from raw tables.')
for name in ATTRIBUTES:
    left = attribute_comparison[f'{name}_original']
    right = attribute_comparison[f'{name}_raw']
    matches = (np.isclose(left.astype(float), right.astype(float), equal_nan=True)
               if name in NUMERIC else left.fillna('<missing>').eq(right.fillna('<missing>')))
    require(np.asarray(matches).all(), f'Raw versus original attributes differ: {name}')
print('Original map-unit attribute comparison passed:', len(attribute_comparison), 'MUKEYs.')

,attribute,label,unit
0,slopegradwta,Slope Gradient - Weighted Average,percent
1,brockdepmin,Bedrock Depth - Minimum,centimeters
2,aws0150wta,Available Water Storage 0-150 cm - Weighted Av...,centimeters
3,drclassdcd,Drainage Class - Dominant Condition,<NA>
4,hydgrpdcd,Hydrologic Group - Dominant Conditions,<NA>


Raw unique map units: 729
Original map-unit attribute comparison passed: 492 MUKEYs.


## 3. Read only polygons near the full buffers

Use a bounding-box filter in each source CRS before loading geometry, then an exact
intersection filter against the combined 1000 m buffers. Reproject each survey before
concatenating. Keep whole source polygons; do not clip them at the watershed boundary.
Explicit topology repair is recorded, and a material area change stops processing.

In [4]:
source_shapes = sorted(RAW_SOILS.glob('*/spatial/soilmu_a_*.shp'))
require(len(source_shapes) == 5, 'Expected the five supplied SSURGO survey shapefiles.')
source_audit = []
soil_layers = []
bounds = buffers[1000].total_bounds
support = buffers[1000].geometry.union_all()
for path in source_shapes:
    info = pyogrio.read_info(path)
    require(info['crs'] is not None, f'Missing source CRS: {path.name}')
    source_bbox = Transformer.from_crs(CRS, info['crs'], always_xy=True).transform_bounds(
        bounds[0]-1, bounds[1]-1, bounds[2]+1, bounds[3]+1, densify_pts=41)
    layer = pyogrio.read_dataframe(path, bbox=source_bbox, fid_as_index=True)
    loaded = len(layer)
    layer['source_id'] = path.stem + ':' + layer.index.astype(str)
    layer = layer.reset_index(drop=True).to_crs(CRS)
    require(layer.geometry.notna().all() and (~layer.is_empty).all(), 'Empty source soil polygons.')
    invalid = ~layer.is_valid
    before = layer.area.sum()
    if invalid.any():
        layer.loc[invalid, 'geometry'] = layer.loc[invalid].geometry.make_valid()
    area_change = float(layer.area.sum()-before)
    require(abs(area_change) < 1, f'Material area change from topology repair: {path.name}')
    require(layer.is_valid.all() and layer.geom_type.isin(['Polygon', 'MultiPolygon']).all(),
            'Soil polygon repair needs manual review.')
    layer = layer[layer.intersects(support)].copy()
    source_audit.append({'survey': path.parent.parent.name, 'source_crs': info['crs'],
                         'source_rows': info['features'], 'bbox_rows': loaded,
                         'selected_polygons': len(layer), 'invalid_repaired': int(invalid.sum()),
                         'repair_area_change_m2': area_change})
    soil_layers.append(layer)
soils = gpd.GeoDataFrame(pd.concat(soil_layers, ignore_index=True), crs=CRS)
soils.MUKEY = soils.MUKEY.astype('string')
require(soils.source_id.is_unique, 'Duplicate source polygon IDs.')
require(not soils.geometry.duplicated().any(), 'Duplicate soil geometry across surveys.')
soils = soils.merge(attributes, on='MUKEY', how='left', validate='many_to_one', indicator=True)
require(soils._merge.eq('both').all(), 'Soil polygons have missing raw attribute records.')
soils = soils.drop(columns='_merge')
display(pd.DataFrame(source_audit))
print('Full soil polygons selected:', len(soils))

,survey,source_crs,source_rows,bbox_rows,selected_polygons,invalid_repaired,repair_area_change_m2
0,CA053,EPSG:4326,14291,12299,2029,0,0.0
1,CA069,EPSG:4326,5426,3980,11,0,0.0
2,CA665,EPSG:4326,7114,6417,2073,0,0.0
3,CA772,EPSG:4326,614,46,4,0,0.0
4,CA788,EPSG:4326,208,208,0,0,0.0


Full soil polygons selected: 4117


## 4. Area-weighted features, with coverage for each attribute

Numeric means use **only area with a known value** as their denominator. Each mean has
a separate coverage percentage relative to the entire geometric buffer. Missing bedrock
depths are not converted to zero. Categorical percentages sum to 100% of the area where
that attribute is known; their coverage columns show how much of the buffer that represents.
Categorical dominant values use the largest known area, with alphabetical tie-breaking.

Unlike the legacy crop percentages, soil coverage uses actual polygon-buffer area as its
denominator, matching the current soil integration. No year or annual change is inferred.

In [5]:
category_values = {name: sorted(attributes[name].dropna().unique()) for name in CATEGORICAL}

def soil_features_for_buffers(circles, polygons, radius):
    pieces = gpd.overlay(circles, polygons[['MUKEY', *ATTRIBUTES, 'geometry']],
                         how='intersection', keep_geom_type=True)
    pieces['area_m2'] = pieces.area
    pieces = pieces[pieces.area_m2.gt(0)].copy()
    ids = pd.Index(circles.gm_well_id, name='gm_well_id')
    full_area = pd.Series(circles.area.to_numpy(), index=ids)
    mapped_union = pieces.groupby('gm_well_id').geometry.apply(lambda s: s.union_all().area)
    mapped_union = mapped_union.reindex(ids, fill_value=0)
    summed_area = pieces.groupby('gm_well_id').area_m2.sum().reindex(ids, fill_value=0)
    overlap = (summed_area-mapped_union).clip(lower=0)
    require(overlap.max() < 1, 'Overlapping soil polygons exceed 1 m2; inspect the source surveys.')
    require((mapped_union <= full_area+0.01).all(), 'Soil coverage exceeds full buffer area.')
    result = pd.DataFrame(index=ids)
    result[f'soil_buffer_area_m2_{radius}m'] = full_area
    result[f'soil_mapped_area_m2_{radius}m'] = mapped_union
    result[f'soil_mapped_coverage_pct_{radius}m'] = mapped_union/full_area*100
    result[f'soil_mapping_status_{radius}m'] = np.where(
        mapped_union.eq(0), 'no_mapped_polygon', np.where(
            mapped_union.ge(full_area-0.01), 'full_mapped_footprint', 'partial_mapped_footprint'))
    for name in ATTRIBUTES:
        valid = pieces[pieces[name].notna()].copy()
        valid_area = valid.groupby('gm_well_id').area_m2.sum().reindex(ids, fill_value=0)
        result[f'soil_{name}_coverage_pct_{radius}m'] = valid_area/full_area*100
        if name in NUMERIC:
            weighted = (valid[name]*valid.area_m2).groupby(valid.gm_well_id).sum()
            result[f'soil_{name}_{radius}m'] = weighted.reindex(ids)/valid_area.replace(0, np.nan)
        else:
            by_class = valid.groupby(['gm_well_id', name], observed=True).area_m2.sum()
            for category in category_values[name]:
                label = str(category).lower().replace(' ', '_').replace('/', '_')
                subset = (by_class.xs(category, level=name) if category in by_class.index.get_level_values(name)
                          else pd.Series(dtype=float))
                result[f'soil_{name}_pct_{label}_{radius}m'] = (
                    subset.reindex(ids, fill_value=0)/valid_area.replace(0, np.nan)*100)
            ranked = by_class.reset_index().sort_values(
                ['gm_well_id', 'area_m2', name], ascending=[True, False, True])
            dominant = ranked.drop_duplicates('gm_well_id').set_index('gm_well_id')[name]
            result[f'soil_{name}_dominant_{radius}m'] = dominant.reindex(ids)
            share_cols = [c for c in result if c.startswith(f'soil_{name}_pct_')]
            totals = result[share_cols].sum(axis=1, min_count=1)
            require(np.allclose(totals[valid_area.gt(0)], 100), 'Category percentages do not sum to 100.')
        require(result[f'soil_{name}_coverage_pct_{radius}m'].between(0, 100+1e-6).all(),
                f'Invalid attribute coverage: {name}')
    require(result.index.is_unique and len(result)==len(circles), 'Soil feature ID/row mismatch.')
    return result, {'radius_m': radius, 'wells': len(result), 'pieces': len(pieces),
                    'max_overlap_m2': float(overlap.max()),
                    'no_mapping_wells': int(mapped_union.eq(0).sum()),
                    'mean_mapped_coverage_pct': float((mapped_union/full_area*100).mean())}

tables, validation = [], []
for radius in RADII:
    table, summary = soil_features_for_buffers(buffers[radius], soils, radius)
    tables.append(table)
    validation.append(summary)
    print('Finished radius:', radius)
soil_features = pd.concat(tables, axis=1).reset_index()
require(soil_features.gm_well_id.is_unique and len(soil_features)==len(wells), 'Incorrect output IDs.')
display(pd.DataFrame(validation))
display(soil_features.head())

Finished radius: 500


Finished radius: 1000


,radius_m,wells,pieces,max_overlap_m2,no_mapping_wells,mean_mapped_coverage_pct
0,500,1993,12250,1.047738e-09,0,100.0
1,1000,1993,27566,3.259629e-09,0,100.0


,gm_well_id,soil_buffer_area_m2_500m,soil_mapped_area_m2_500m,soil_mapped_coverage_pct_500m,soil_mapping_status_500m,soil_slopegradwta_coverage_pct_500m,soil_slopegradwta_500m,soil_brockdepmin_coverage_pct_500m,soil_brockdepmin_500m,soil_aws0150wta_coverage_pct_500m,...,soil_drclassdcd_pct_well_drained_1000m,soil_drclassdcd_dominant_1000m,soil_hydgrpdcd_coverage_pct_1000m,soil_hydgrpdcd_pct_a_1000m,soil_hydgrpdcd_pct_a_d_1000m,soil_hydgrpdcd_pct_b_1000m,soil_hydgrpdcd_pct_c_1000m,soil_hydgrpdcd_pct_c_d_1000m,soil_hydgrpdcd_pct_d_1000m,soil_hydgrpdcd_dominant_1000m
0,AGL020000465-PRIMARY,784137.122636,784137.122636,100.0,full_mapped_footprint,100.0,13.944009,50.365606,56.466397,100.0,...,100.000000,Well drained,10.843366,0.000000,0.0,0.000000,100.000000,0.0,0.0,C
1,AGL020000484-ROOS-HOMESTEAD,784137.122636,784137.122636,100.0,full_mapped_footprint,100.0,22.068256,32.604915,41.000000,100.0,...,100.000000,Well drained,55.500217,3.762210,0.0,0.008456,96.229334,0.0,0.0,C
2,AGL020000508-AG WELL,784137.122636,784137.122636,100.0,full_mapped_footprint,100.0,19.849474,84.509763,41.000000,100.0,...,100.000000,Well drained,17.077412,0.000000,0.0,0.000000,100.000000,0.0,0.0,C
3,AGL020000508-DOMESTIC,784137.122636,784137.122636,100.0,full_mapped_footprint,100.0,18.484246,80.263060,41.036725,100.0,...,100.000000,Well drained,21.001925,0.000000,0.0,0.000000,100.000000,0.0,0.0,C
4,AGL020000518-M-24,784137.122636,784137.122636,100.0,full_mapped_footprint,100.0,1.520223,0.000000,NaN,100.0,...,58.416911,Well drained,100.000000,25.528586,0.0,32.140608,42.330806,0.0,0.0,C


## 5. Compare with the original polygon output

First, all original map-unit values were checked against the raw tables above.
Next, compare five interior wells and one edge well at each radius. Interior buffers
must match the original clipped layer numerically. An edge buffer may differ because
Andrea retains full buffers and soil polygons beyond the HUC boundary; show that difference
instead of treating clipped coverage as complete. This does not alter the original layer.

In [6]:
CROPS_RAW = RAW_CROPS
boundary_paths = [p.parent for p in CROPS_RAW.rglob('a00000001.gdbtable')
                  if p.parent.name == 'NHD_H_18060005_HU8_GDB.gdb']
require(len(boundary_paths)==1, 'Missing or ambiguous Salinas boundary.')
boundary = gpd.read_file(boundary_paths[0], layer='WBDHU8',
                         where="huc8 = '18060005'").to_crs(CRS)
boundary_shape = boundary.geometry.union_all()
inside = buffers[1000].geometry.covered_by(boundary_shape)
inside_ids = buffers[1000].loc[inside, 'gm_well_id'].iloc[
    np.linspace(0, int(inside.sum())-1, 5, dtype=int)].tolist()
edge_ids = buffers[1000].loc[~inside, 'gm_well_id'].head(1).tolist()
example_ids = inside_ids + edge_ids
original_polygons = gpd.read_file(ORIGINAL_GPKG).to_crs(CRS)
require(original_polygons.is_valid.all(), 'Original soil polygons need review before comparison.')
comparison_rows = []
for radius in RADII:
    example_buffers = buffers[radius][buffers[radius].gm_well_id.isin(example_ids)]
    reference, _ = soil_features_for_buffers(example_buffers, original_polygons, radius)
    current = soil_features.set_index('gm_well_id').loc[reference.index]
    for well_id in example_ids:
        for name in NUMERIC:
            column = f'soil_{name}_{radius}m'
            a, b = current.loc[well_id, column], reference.loc[well_id, column]
            matches = bool(np.isclose(a, b, atol=1e-6, rtol=1e-8, equal_nan=True))
            if well_id in inside_ids:
                require(matches, f'Interior soil feature disagrees with original: {well_id}, {column}')
            comparison_rows.append({'gm_well_id': well_id, 'radius_m': radius, 'attribute': name,
                                    'buffer_1000m_inside_huc': well_id in inside_ids,
                                    'andrea_full_buffer': a, 'original_huc_clip': b,
                                    'matches': matches})
comparison = pd.DataFrame(comparison_rows)
display(comparison)
print('Interior examples passed:', len(inside_ids), 'wells at both radii.')

,gm_well_id,radius_m,attribute,buffer_1000m_inside_huc,andrea_full_buffer,original_huc_clip,matches
0,AGL020000465-PRIMARY,500,slopegradwta,True,13.944009,13.944009,True
1,AGL020000465-PRIMARY,500,brockdepmin,True,56.466397,56.466397,True
2,AGL020000465-PRIMARY,500,aws0150wta,True,15.618785,15.618785,True
3,AGL020003659-AG_FERRAS1,500,slopegradwta,True,1.000000,1.000000,True
4,AGL020003659-AG_FERRAS1,500,brockdepmin,True,NaN,NaN,True
5,AGL020003659-AG_FERRAS1,500,aws0150wta,True,23.110189,23.110189,True
6,AGL020011789-WELL AG 1,500,slopegradwta,True,1.000000,1.000000,True
7,AGL020011789-WELL AG 1,500,brockdepmin,True,NaN,NaN,True
8,AGL020011789-WELL AG 1,500,aws0150wta,True,23.115673,23.115673,True
9,AGL020035970-TUNNEL 108,500,slopegradwta,True,2.812272,2.812272,True


Interior examples passed: 5 wells at both radii.


## 6. Validate the nitrate join and save Andrea's CSV only

No nitrate values are aggregated or changed. The soil table is unique by well ID, so the
same static soil features can attach to every measurement at that well without adding rows.
Keep missingness and coverage for review before deciding which columns should enter ML.
Do not automatically fill missing soil values with zero or use missing bedrock as a depth.

In [7]:
joined_check = observations.merge(soil_features, on='gm_well_id', how='left',
                                   validate='many_to_one', sort=False, indicator=True)
require(len(joined_check)==len(observations) and joined_check._merge.eq('both').all(),
        'Soil join changed nitrate rows or missed well IDs.')
pd.testing.assert_frame_equal(joined_check[observations.columns], observations)
require(protected_hashes == {p: file_hash(p) for p in protected_hashes},
        'An original input changed during processing; review before saving.')
display(soil_features.isna().mean().sort_values(ascending=False).to_frame('missing_fraction').head(12))
print('Validation passed:', len(soil_features), 'wells;', len(joined_check), 'nitrate rows preserved.')
print('All original notebook/output hashes are unchanged.')

# Exclusive creation preserves any existing output, including Andrea's earlier runs.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('x', encoding='utf-8', newline='') as stream:
    soil_features.to_csv(stream, index=False)
saved = pd.read_csv(OUTPUT_PATH, dtype={'gm_well_id':'string'})
pd.testing.assert_frame_equal(saved, soil_features, check_dtype=False,
                              check_exact=False, atol=1e-10, rtol=1e-10)
print('Saved:', OUTPUT_PATH)

,missing_fraction
soil_brockdepmin_500m,0.697441
soil_brockdepmin_1000m,0.624185
soil_hydgrpdcd_pct_a_500m,0.007526
soil_hydgrpdcd_pct_c_500m,0.007526
soil_hydgrpdcd_pct_c_d_500m,0.007526
soil_hydgrpdcd_pct_b_500m,0.007526
soil_hydgrpdcd_pct_a_d_500m,0.007526
soil_hydgrpdcd_pct_d_500m,0.007526
soil_hydgrpdcd_dominant_500m,0.007526
soil_hydgrpdcd_pct_c_1000m,0.001505


Validation passed: 1993 wells; 5280 nitrate rows preserved.
All original notebook/output hashes are unchanged.


Saved: C:\Users\lomel\.codex\.chatgpt-projects\g-p-6abbf6fb87108191989129cda2477ef0\soil_features_andrea.csv


C:\Users\lomel\AppData\Local\Temp\ipykernel_35596\1035970680.py:17: FutureWarning: Mismatched null-like values nan and <NA> found. In a future version, pandas equality-testing functions (e.g. assert_frame_equal) will consider these not-matching and raise.
  pd.testing.assert_frame_equal(saved, soil_features, check_dtype=False,


## Review before integration

Compare the saved features, attribute coverage and the original-output examples above.
The original CSV is polygon-level (15,906 rows), while Andrea's output is well-level
(1,993 rows): row counts are not expected to match. Five raw surveys cover this study area;
remaining spatial or attribute gaps stay visible in the coverage columns.

For a future merge, read `gm_well_id` as text and use a left join with
`validate='many_to_one'`. The main integration, Gavin's original notebook and all of its
outputs remain independent. No commit or push is performed by this notebook.